# D2 · Diagnosing missingness mechanisms: what you can and cannot know

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

For each column with missing values in the loans dataset, establish with statistical tests whether the missingness is compatible
with MCAR or depends on other variables (MAR), and say what the data **cannot** prove.

## Dataset

`ds.loans()` (with missing values): columns with missing values `age`, `income`, `job_tenure`, `education`, `occupation`. The missingness
mechanisms were inserted differently for each one and are hidden on purpose. The other columns, including the
target `defaulted`, are observed (with a few missing values themselves).

## Educational purpose

Know **how far** the data let you investigate the missingness mechanism: you can detect dependencies on the observed
variables (MAR), but you cannot prove MCAR, and you cannot even detect MNAR by looking only at the observed ones.

## Task

1. Write `dependence_test(df, col)`: for the column `col` build the indicator `m = df[col].isna()` and, for **every other column** `c`, compare `c` between the rows with `col` missing and those with `col` present, using only the rows where `c` is known. For numeric columns (including `defaulted`) use `scipy.stats.mannwhitneyu`; for categorical ones `scipy.stats.chi2_contingency` on `pd.crosstab(m, df[c])`. Return `(corrected_min_p, column)`: the smallest p-value multiplied by the number of tests (Bonferroni correction, at most 1) and the name of the column that produced it.
2. For every column with missing values compute `corrected_min_p` and fill the DataFrame `table` (index: columns with missing values; columns: `missing_share`, `corrected_p`, `depends_on`).
3. Assign to `mechanism` a dictionary `{column: "MCAR" | "MAR"}`: `"MAR"` if `corrected_p < 0.001`, otherwise `"MCAR"`.
4. Compute `default_if_income_missing` and `default_if_income_present`: the default rate with the income missing and with the income known.
5. Assign to `true_false` a dictionary:
   - **a**: a non-significant test proves that the missingness of `age` is MCAR;
   - **b**: if the missingness depended on the value of income itself (MNAR), these tests would always detect it;
   - **c**: the fact that `income` is missing is useful information for predicting `defaulted`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats

df = ds.loans()
with_missing = [c for c in df.columns if df[c].isna().any()]
df[with_missing].isna().mean().round(3)

## Your work

In [ ]:
def dependence_test(df, col):
    m = df[col].isna()
    p_values = {}
    for c in df.columns:
        if c == col:
            continue
        ...
    ...
    return corrected_min_p, column

In [ ]:
rows = {}
for col in with_missing:
    p, by = dependence_test(df, col)
    rows[col] = {"missing_share": df[col].isna().mean(), "corrected_p": p, "depends_on": by}
table = pd.DataFrame(rows).T
mechanism = ...
table

In [ ]:
default_if_income_missing = ...
default_if_income_present = ...
true_false = {"a": ..., "b": ..., "c": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For every column `c`: `ok = df[c].notna()`; if numeric `stats.mannwhitneyu(df.loc[ok & m, c], df.loc[ok & ~m, c]).pvalue`; if categorical `stats.chi2_contingency(pd.crosstab(m[ok], df.loc[ok, c]))[1]`. Numeric: `pd.api.types.is_numeric_dtype(df[c])`. Use the minimum of the p-values for the corrected p-value: `min(1, p_min * len(p_values))`.'):
    p, by = dependence_test(df, "income")
    check(0 <= p <= 1 and by == "defaulted", "for 'income' the strongest dependence is with the target defaulted",
          "income is missing much more often among defaulters: the test on the target must have the minimum p-value")
    p2, by2 = dependence_test(df, "job_tenure")
    check(by2 == "contract_type" and p2 < 1e-6, "'job_tenure' depends on the contract type", "the missingness depends on contract_type")
    check(list(table.columns) == ["missing_share", "corrected_p", "depends_on"] and set(table.index) == set(with_missing),
          "table with one row per column and the three required columns", "columns: missing_share, corrected_p, depends_on")
    check(mechanism == {"age": "MCAR", "income": "MAR", "job_tenure": "MAR", "education": "MCAR", "occupation": "MCAR"},
          "mechanisms diagnosed correctly", "MAR where corrected_p < 0.001, MCAR elsewhere")
    check(default_if_income_missing > 3 * default_if_income_present, "whoever does not declare income defaults much more often (over 3 times)",
          "compare the default rate between rows with income missing and present")
    check(true_false == {"a": False, "b": False, "c": True}, "statements judged correctly",
          "a non-significant test proves nothing; MNAR is not detectable from the observed ones; the missingness is informative")
done()

---
The commented solution is in `solutions/D_interpret/D2_missingness_mechanisms.ipynb`. Open it after you pass the check, or after you have really tried.